In [19]:
# ======================================
# Initialize Spark Session (Fabric)
# ======================================

from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

spark = SparkSession.builder.getOrCreate()

print("Spark initialized")

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 25, Finished, Available, Finished, False)

Spark initialized


In [20]:
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 26, Finished, Available, Finished, False)

+---------------------------------------+---------------+-----------+
|namespace                              |tableName      |isTemporary|
+---------------------------------------+---------------+-----------+
|SmogNet_Datathon.Datathon_Lakehouse.dbo|bronze_testing |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|bronze_training|false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|silver_testing |false      |
|SmogNet_Datathon.Datathon_Lakehouse.dbo|silver_training|false      |
+---------------------------------------+---------------+-----------+



In [21]:
train = spark.sql(
    "SELECT * FROM Bronze_training"
).toPandas()

test = spark.sql(
    "SELECT * FROM Bronze_testing"
).toPandas()

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

train.head()

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 27, Finished, Available, Finished, False)

Train Shape: (264335, 20)
Test Shape: (21792, 20)


,datetime,main_aqi,components_co,components_no,components_no2,components_o3,components_so2,components_pm2_5,components_pm10,components_nh3,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m,shortwave_radiation,dataset_type,source_file
0,10/20/2023 5:00,5,6355.29,106.39,146.69,28.25,21.93,233.53,264.28,42.56,15.4,74,10.8,0,955.7,3.3,13,0,Training,abfss://80ba3f90-becf-431a-a98d-cfd1895b7c09@o...
1,10/20/2023 6:00,5,5874.63,60.8,167.25,69.38,19.55,252.88,281.8,31.66,15.2,72,10.1,0,955.7,2.9,360,0,Training,abfss://80ba3f90-becf-431a-a98d-cfd1895b7c09@o...
2,10/20/2023 7:00,5,1575.47,5.7,40.1,175.95,30.99,103.66,114.17,10.77,15.1,72,10.1,0,956.1,3.3,347,27,Training,abfss://80ba3f90-becf-431a-a98d-cfd1895b7c09@o...
3,10/20/2023 8:00,5,1001.36,2.01,17.82,203.13,26.7,79.16,87.03,8.36,16,74,11.4,0,957.1,1.9,338,172,Training,abfss://80ba3f90-becf-431a-a98d-cfd1895b7c09@o...
4,10/20/2023 9:00,5,921.25,1.37,14.91,214.58,25.51,75.97,83.38,8.23,19.6,62,12.1,0,957.9,1,225,357,Training,abfss://80ba3f90-becf-431a-a98d-cfd1895b7c09@o...


In [22]:
train["parsed_datetime"] = pd.to_datetime(
    train["datetime"],
    errors="coerce",
    format="mixed"
)

test["parsed_datetime"] = pd.to_datetime(
    test["datetime"],
    errors="coerce",
    format="mixed"
)

print(
    train["parsed_datetime"].isna().sum()
)

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 29, Finished, Available, Finished, False)

0


In [23]:
for df in [train,test]:

    df["hour"]=df[
        "parsed_datetime"
    ].dt.hour

    df["day"]=df[
        "parsed_datetime"
    ].dt.day

    df["month"]=df[
        "parsed_datetime"
    ].dt.month

    df["year"]=df[
        "parsed_datetime"
    ].dt.year

    df["weekday"]=df[
        "parsed_datetime"
    ].dt.dayofweek

    df["weekofyear"]=df[
        "parsed_datetime"
    ].dt.isocalendar().week

    df["is_weekend"]=(
        df["weekday"]>=5
    ).astype(int)

train.head()

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 30, Finished, Available, Finished, False)

,datetime,main_aqi,components_co,components_no,components_no2,components_o3,components_so2,components_pm2_5,components_pm10,components_nh3,...,dataset_type,source_file,parsed_datetime,hour,day,month,year,weekday,weekofyear,is_weekend
0,10/20/2023 5:00,5,6355.29,106.39,146.69,28.25,21.93,233.53,264.28,42.56,...,Training,abfss://80ba3f90-becf-431a-a98d-cfd1895b7c09@o...,2023-10-20 05:00:00,5,20,10,2023,4,42,0
1,10/20/2023 6:00,5,5874.63,60.8,167.25,69.38,19.55,252.88,281.8,31.66,...,Training,abfss://80ba3f90-becf-431a-a98d-cfd1895b7c09@o...,2023-10-20 06:00:00,6,20,10,2023,4,42,0
2,10/20/2023 7:00,5,1575.47,5.7,40.1,175.95,30.99,103.66,114.17,10.77,...,Training,abfss://80ba3f90-becf-431a-a98d-cfd1895b7c09@o...,2023-10-20 07:00:00,7,20,10,2023,4,42,0
3,10/20/2023 8:00,5,1001.36,2.01,17.82,203.13,26.7,79.16,87.03,8.36,...,Training,abfss://80ba3f90-becf-431a-a98d-cfd1895b7c09@o...,2023-10-20 08:00:00,8,20,10,2023,4,42,0
4,10/20/2023 9:00,5,921.25,1.37,14.91,214.58,25.51,75.97,83.38,8.23,...,Training,abfss://80ba3f90-becf-431a-a98d-cfd1895b7c09@o...,2023-10-20 09:00:00,9,20,10,2023,4,42,0


In [24]:
def get_season(month):

    if month in [12,1,2]:
        return "Winter"

    elif month in [3,4,5]:
        return "Spring"

    elif month in [6,7,8]:
        return "Summer"

    return "Autumn"


train["season"]=train[
    "month"
].apply(get_season)

test["season"]=test[
    "month"
].apply(get_season)

train[
["month","season"]
].head()

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 31, Finished, Available, Finished, False)

,month,season
0,10,Autumn
1,10,Autumn
2,10,Autumn
3,10,Autumn
4,10,Autumn


In [25]:
features=[

"components_co",
"components_no",
"components_no2",
"components_o3",
"components_so2",
"components_pm2_5",
"components_pm10",
"components_nh3",

"temperature_2m",
"relative_humidity_2m",
"precipitation",
"surface_pressure",
"wind_speed_10m"

]

train[features]=train[
features
].ffill()

test[features]=test[
features
].ffill()

print("Missing values fixed")

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 32, Finished, Available, Finished, False)

Missing values fixed


In [26]:
for col in [

"components_pm2_5",
"components_pm10",
"components_no2"

]:

    train[col+"_roll_mean"]=train[col].rolling(24).mean()

    train[col+"_roll_std"]=train[col].rolling(24).std()


    test[
        col+"_roll_mean"
    ]=test[
        col
    ].rolling(
        24
    ).mean()

    test[
        col+"_roll_std"
    ]=test[
        col
    ].rolling(
        24
    ).std()

print("Rolling features created")

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 33, Finished, Available, Finished, False)

Rolling features created


In [27]:
for c in [

"components_pm2_5",
"components_pm10"

]:

    train[
        c+"_lag1"
    ]=train[
        c
    ].shift(1)

    train[
        c+"_lag24"
    ]=train[
        c
    ].shift(24)


    test[
        c+"_lag1"
    ]=test[
        c
    ].shift(1)

    test[
        c+"_lag24"
    ]=test[
        c
    ].shift(24)

print("Lag complete")

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 34, Finished, Available, Finished, False)

Lag complete


In [28]:
# ======================================
# Convert pollutant columns to numeric
# ======================================

pollution_cols=[

"components_pm10",
"components_pm2_5",
"components_no",
"components_no2",
"components_so2",
"components_co"

]

for df in [train,test]:

    for c in pollution_cols:

        df[c]=pd.to_numeric(
            df[c],
            errors="coerce"
        )

    # Avoid divide by zero

    df["PM10_PM25_ratio"]=np.where(

        df["components_pm2_5"]>0,

        df["components_pm10"]/
        df["components_pm2_5"],

        0

    )

    df["NO2_NO_ratio"]=np.where(

        df["components_no"]>0,

        df["components_no2"]/
        df["components_no"],

        0

    )

    df["SO2_CO_ratio"]=np.where(

        df["components_co"]>0,

        df["components_so2"]/
        df["components_co"],

        0

    )

print("Pollution ratios created")

train[
[
"PM10_PM25_ratio",
"NO2_NO_ratio",
"SO2_CO_ratio"
]
].head()

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 35, Finished, Available, Finished, False)

Pollution ratios created


,PM10_PM25_ratio,NO2_NO_ratio,SO2_CO_ratio
0,1.131675,1.378795,0.003451
1,1.114363,2.750822,0.003328
2,1.101389,7.035088,0.019670
3,1.099419,8.865672,0.026664
4,1.097539,10.883212,0.027691


In [29]:
# ======================================
# Severity Score
# ======================================

severity_cols=[

"components_pm2_5",
"components_pm10",
"main_aqi"

]

for df in [train,test]:

    # Convert columns to numeric
    for c in severity_cols:

        df[c]=pd.to_numeric(
            df[c],
            errors="coerce"
        )

    # Fill missing values
    df[severity_cols]=df[
        severity_cols
    ].fillna(0)

    # Severity calculation
    df["severity_score"]=(
        (df["components_pm2_5"]*0.4)+
        (df["components_pm10"]*0.3)+
        (df["main_aqi"]*0.3)
    )

print("Severity score created")

train[
[
"components_pm2_5",
"components_pm10",
"main_aqi",
"severity_score"
]
].head()

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 36, Finished, Available, Finished, False)

Severity score created


,components_pm2_5,components_pm10,main_aqi,severity_score
0,233.53,264.28,5,174.196
1,252.88,281.80,5,187.192
2,103.66,114.17,5,77.215
3,79.16,87.03,5,59.273
4,75.97,83.38,5,56.902


In [30]:
# ======================================
# Fix unsupported Arrow datatypes
# ======================================

for df in [train,test]:

    # convert weekofyear
    if "weekofyear" in df.columns:

        df["weekofyear"]=(
            df["weekofyear"]
            .astype("int64")
        )

    # convert weekend flag
    if "is_weekend" in df.columns:

        df["is_weekend"]=(
            df["is_weekend"]
            .astype("int64")
        )

print("Datatype conversion complete")

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 37, Finished, Available, Finished, False)

Datatype conversion complete


In [31]:
# ==========================================
# Final datatype cleaning for ML
# ==========================================

numeric_cols=[

"components_co",
"components_no",
"components_no2",
"components_o3",
"components_so2",
"components_pm2_5",
"components_pm10",
"components_nh3",

"temperature_2m",
"relative_humidity_2m",
"dew_point_2m",
"precipitation",
"surface_pressure",
"wind_speed_10m",
"wind_direction_10m",
"shortwave_radiation",

"components_pm2_5_lag1",
"components_pm2_5_lag24",
"components_pm10_lag1",
"components_pm10_lag24",

"components_pm2_5_roll_mean",
"components_pm2_5_roll_std",

"components_pm10_roll_mean",
"components_pm10_roll_std",

"components_no2_roll_mean",
"components_no2_roll_std",

"PM10_PM25_ratio",
"NO2_NO_ratio",
"SO2_CO_ratio",

"severity_score"

]

for df in [train,test]:

    for c in numeric_cols:

        df[c]=pd.to_numeric(
            df[c],
            errors="coerce"
        )

    # Fill rolling/lag NaNs
    df[numeric_cols]=df[
        numeric_cols
    ].fillna(0)

print("All datatypes corrected")

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 38, Finished, Available, Finished, False)

All datatypes corrected


In [33]:
silver_train=spark.createDataFrame(train)
silver_test=spark.createDataFrame(test)

silver_train.write.mode("overwrite").format("delta").saveAsTable("Silver_training")

silver_test.write.mode("overwrite").format("delta").saveAsTable("Silver_testing")

print("Silver layer created successfully")

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 41, Finished, Available, Finished, False)

Silver layer created successfully


In [34]:
display(
spark.sql("""

SELECT
parsed_datetime,
season,
severity_score
FROM Silver_training
LIMIT 20

""")
)

StatementMeta(, 81a8df12-8e65-402f-9637-7287cd9cba96, 42, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9d89d140-3228-4bc9-96a9-edf806192b58)